# IPL First Innings Score Prediction — Advanced

**Tabular Regression Project** — Predict first innings cricket scores.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: IPL match data (local `dataset/ipl.csv`)
Target: `total` (first innings score)

## 2 · Project Overview

We predict the **first innings total score** in Indian Premier League (IPL) cricket matches. Given the batting team, bowling team, venue, current runs, wickets, overs, and recent run rates, we predict the final score. This is a popular sports analytics regression problem.

## 3 · Learning Objectives

1. Work with sports analytics data.
2. Handle cricket-specific features (overs, wickets, run rate).
3. Apply regression to live score prediction.
4. Use LazyPredict and FLAML.
5. Understand domain-specific feature engineering.

## 4 · Problem Statement

Predict the **total first innings score** at the end of 20 overs, given match context at any point during the innings.

## 5 · Why This Project Matters

- **Sports prediction** is a high-engagement ML application.
- IPL is the world's most valuable cricket league.
- Live score prediction enhances fan experience and betting analytics.
- Teaches domain-specific feature engineering.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local: `dataset/ipl.csv` |
| **Target** | total (first innings score) |
| **Features** | bat_team, bowl_team, venue, runs, wickets, overs, runs_last_5, wickets_last_5, etc. |

## 7 · Dataset Source and License Notes

- **Source**: IPL ball-by-ball data, commonly available on Kaggle.
- **License**: Educational use.
- **Note**: Historical IPL match data covering multiple seasons.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
np.random.seed(SEED)

## 11 · Dataset Loading

In [4]:
data_file = os.path.join(SAVE_DIR, 'dataset', 'ipl.csv')
assert os.path.exists(data_file), f'Not found: {data_file}'
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Loaded: (76014, 15)
Columns: ['mid', 'date', 'venue', 'bat_team', 'bowl_team', 'batsman', 'bowler', 'runs', 'wickets', 'overs', 'runs_last_5', 'wickets_last_5', 'striker', 'non-striker', 'total']


,mid,date,venue,bat_team,bowl_team,batsman,bowler,runs,wickets,overs,runs_last_5,wickets_last_5,striker,non-striker,total
0,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,P Kumar,1,0,0.1,1,0,0,0,222
1,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,1,0,0.2,1,0,0,0,222
2,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.2,2,0,0,0,222
3,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.3,2,0,0,0,222
4,1,2008-04-18,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,P Kumar,2,0,0.4,2,0,0,0,222


## 12 · Data Validation

In [5]:
print('Missing:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

Missing:
mid               0
date              0
venue             0
bat_team          0
bowl_team         0
batsman           0
bowler            0
runs              0
wickets           0
overs             0
runs_last_5       0
wickets_last_5    0
striker           0
non-striker       0
total             0
dtype: int64



Duplicates: 0


## 13 · Exploratory Data Analysis

In [6]:
# Find target
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['total', 'score', 'target'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    TARGET = num_cols[-1]
print(f'Target: {TARGET}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=40, ax=axes[0], edgecolor='black')
axes[0].set_title(f'{TARGET} Distribution')
if 'bat_team' in df.columns:
    df.groupby('bat_team')[TARGET].mean().sort_values().tail(10).plot.barh(ax=axes[1])
    axes[1].set_title('Avg Score by Batting Team')
elif 'batting_team' in df.columns:
    df.groupby('batting_team')[TARGET].mean().sort_values().tail(10).plot.barh(ax=axes[1])
    axes[1].set_title('Avg Score by Batting Team')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

Target: total


## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count    76014.000000
mean       160.901452
std         29.246231
min         67.000000
25%        142.000000
50%        162.000000
75%        181.000000
max        263.000000
Name: total, dtype: float64

Skewness: -0.06


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder

# Drop date columns
for col in df.columns:
    if 'date' in col.lower() or 'mid' in col.lower():
        df = df.drop(columns=[col])

# Encode categoricals
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 100:
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Preprocessed: (76014, 11)


## 17 · Feature Engineering

In [9]:
# Add run rate if overs available
for overs_col in ['overs', 'over']:
    if overs_col in df.columns:
        runs_col = [c for c in df.columns if 'run' in c.lower() and c != TARGET]
        if runs_col:
            df['current_rr'] = df[runs_col[0]] / (df[overs_col] + 0.1)
        break

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Clean inf/nan
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    X[col] = X[col].fillna(X[col].median() if pd.notna(X[col].median()) else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (60811, 11), Test: (15203, 11)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=20.04, MAE=14.78, R²=0.5239


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared       RMSE  \
Model                                                                     
XGBRegressor                             0.696635   0.699976  15.744304   
CatBoostRegressor                        0.695352   0.698706  15.777578   
ExtraTreesRegressor                      0.665977   0.669655  16.520727   
LGBMRegressor                            0.659900   0.663645  16.670322   
HistGradientBoostingRegressor            0.659354   0.663105  16.683715   
RandomForestRegressor                    0.635573   0.639585  17.256249   
BaggingRegressor                         0.599391   0.603802  18.092621   
GradientBoostingRegressor                0.571944   0.576657  18.702140   
MLPRegressor                             0.534559   0.539684  19.501736   
OrthogonalMatchingPursuitCV              0.531182   0.536344  19.572363   
SGDRegressor                             0.527595   0.532797  19.647082   
LassoCV                  

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=14.76, MAE=10.64, R²=0.7417


LightGBM: RMSE=12.68, MAE=8.89, R²=0.8095


XGBoost: RMSE=11.55, MAE=7.91, R²=0.8420


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  XGBoost               RMSE=11.55  MAE=7.91  R²=0.8420
  LightGBM              RMSE=12.68  MAE=8.89  R²=0.8095
  CatBoost              RMSE=14.76  MAE=10.64  R²=0.7417
  Baseline_LR           RMSE=20.04  MAE=14.78  R²=0.5239

Top 3: ['XGBoost', 'LightGBM', 'CatBoost']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
XGBoost: RMSE=11.55, MAE=7.91, R²=0.8420
LightGBM: RMSE=12.68, MAE=8.89, R²=0.8095
CatBoost: RMSE=14.76, MAE=10.64, R²=0.7417


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Current runs** and **overs bowled** are the strongest predictors (as expected).
- **Wickets lost** constrains scoring potential — more wickets = lower final score.
- **Runs in last 5 overs** captures momentum.
- **Team identity** matters — some teams consistently score higher.
- Useful for live match score projection and fantasy cricket.

## 26 · Limitations

- Historical data doesn't account for team roster changes.
- Pitch conditions and weather not captured.
- Player-level features (batsman form) not included.
- Model assumes relatively stable cricket conditions.

## 27 · How to Improve

1. Add player-level features (batsman strike rate, bowler economy).
2. Include pitch/weather data.
3. Add powerplay/death overs phase indicator.
4. Use sequential models for over-by-over prediction.

## 28 · Production Considerations

- Real-time data integration needed.
- Low-latency inference for live prediction.
- Gambling regulation compliance.
- Regular retraining with each new season.

## 29 · Common Mistakes

1. Including target-leaking features (final score components).
2. Not creating run rate features.
3. Ignoring the phase of innings (powerplay vs death).
4. Using player names directly (too many categories).

## 30 · Mini Challenge

1. Predict only at the 10-over mark (mid-innings).
2. Add a 'batting_powerplay' binary feature.
3. Compare predictions at different over stages.
4. Build a model using only the first 6 overs' data.

## 31 · Final Summary

- IPL score prediction is an engaging sports analytics regression task.
- Domain knowledge (cricket phases, run rates) drives feature engineering.
- Current match state (runs, wickets, overs) dominates predictions.
- Gradient-boosting models handle the team/venue categorical features well.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
